# MT-SimNPO Analysis

Results from cloud training runs on RunPod A40 48GB.  
Models: Llama-3.1-8B-Instruct fine-tuned on TOFU (forget10 split).

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', font_scale=1.2)
FIGDIR = Path('figures')
FIGDIR.mkdir(exist_ok=True)

## 1. Data

In [ ]:
# TOFU benchmark results (forget10 split, seed=0 for baselines)
# forget_truth_ratio: ↑ better (closer to oracle_retrain ~0.64 is good)
# model_utility:      ↑ better
# Higher forget_truth_ratio = model behaves more like it never saw the forget set.

tofu = {
    'oracle_retrain':    {'ftr': 0.6406, 'mu': 0.6471},
    'GradDiff':          {'ftr': 0.0000, 'mu': 0.6713},
    'SimNPO':            {'ftr': 0.5230, 'mu': 0.6365},
    'NPO':               {'ftr': None,   'mu': None},     # training in progress
    'MTSimNPO (mw=0.5)': {'ftr': 0.5242, 'mu': None},
    'MTSimNPO (mw=1.0)': {'ftr': np.mean([0.5315, 0.5266, 0.5229]), 'mu': None},
    'MTSimNPO (mw=2.0)': {'ftr': 0.5241, 'mu': None},
}

# MT-Eval results — Multi-Turn Recovery Rate (MTRR, ↓ better)
# Evaluated on mt_val.jsonl (3 trained attack types: priming_v2, self_correction_v2, persona_switch_v2)
mt_eval = {
    'MTSimNPO (mw=0.5)':  {'mtrr': 0.7075, 'seeds': [0]},
    'MTSimNPO (mw=1.0)':  {'mtrr': np.mean([0.69, 0.735, 0.6675]), 'seeds': [0.69, 0.735, 0.6675]},
    'MTSimNPO (mw=2.0)':  {'mtrr': 0.615,  'seeds': [0]},
}

print('TOFU Results:')
for k, v in tofu.items():
    ftr = f"{v['ftr']:.4f}" if v['ftr'] is not None else 'TBD'
    mu  = f"{v['mu']:.4f}"  if v['mu']  is not None else 'TBD'
    print(f"  {k:<25} ftr={ftr}  mu={mu}")

print('\nMT-Eval Results (MTRR, lower=better):')
for k, v in mt_eval.items():
    print(f"  {k:<25} mtrr={v['mtrr']:.4f}")

## 2. TOFU: Forget Quality vs Model Utility

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

colors = {
    'oracle_retrain':    '#2ca02c',
    'GradDiff':          '#d62728',
    'SimNPO':            '#ff7f0e',
    'MTSimNPO (mw=0.5)': '#9467bd',
    'MTSimNPO (mw=1.0)': '#1f77b4',
    'MTSimNPO (mw=2.0)': '#17becf',
}
markers = {
    'oracle_retrain':    '*',
    'GradDiff':          's',
    'SimNPO':            'D',
    'MTSimNPO (mw=0.5)': 'o',
    'MTSimNPO (mw=1.0)': 'o',
    'MTSimNPO (mw=2.0)': 'o',
}

for name, vals in tofu.items():
    if vals['ftr'] is None or vals['mu'] is None:
        continue
    ax.scatter(vals['mu'], vals['ftr'],
               color=colors.get(name, 'gray'),
               marker=markers.get(name, 'o'),
               s=150, zorder=5, label=name)
    ax.annotate(name, (vals['mu'], vals['ftr']),
                textcoords='offset points', xytext=(6, 4), fontsize=9)

# Oracle reference lines
oracle_ftr = tofu['oracle_retrain']['ftr']
oracle_mu  = tofu['oracle_retrain']['mu']
ax.axhline(oracle_ftr, color='#2ca02c', ls='--', lw=1, alpha=0.5, label='Oracle FTR')
ax.axvline(oracle_mu,  color='#2ca02c', ls='--', lw=1, alpha=0.5, label='Oracle MU')

ax.set_xlabel('Model Utility (↑)')
ax.set_ylabel('Forget Truth Ratio (↑ toward oracle)')
ax.set_title('TOFU: Forget Quality vs Utility')
ax.legend(fontsize=8, loc='lower right')
plt.tight_layout()
plt.savefig(FIGDIR / 'tofu_scatter.png', dpi=150)
plt.show()

## 3. TOFU: Forget Truth Ratio Bar Chart

In [ ]:
methods_bar = ['GradDiff', 'SimNPO', 'MTSimNPO (mw=0.5)', 'MTSimNPO (mw=1.0)', 'MTSimNPO (mw=2.0)']
ftr_vals = [tofu[m]['ftr'] for m in methods_bar]
bar_colors = [colors[m] for m in methods_bar]

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(methods_bar, ftr_vals, color=bar_colors, edgecolor='white', width=0.6)

# MTSimNPO mw=1.0 seed error bar
mw1_idx = methods_bar.index('MTSimNPO (mw=1.0)')
mw1_seeds = [0.5315, 0.5266, 0.5229]
mw1_std = np.std(mw1_seeds)
ax.errorbar(mw1_idx, np.mean(mw1_seeds), yerr=mw1_std,
            fmt='none', color='black', capsize=5, capthick=2, lw=2)

ax.axhline(oracle_ftr, color='#2ca02c', ls='--', lw=2, label=f'Oracle retrain ({oracle_ftr:.3f})')
ax.set_ylabel('Forget Truth Ratio (↑ better)')
ax.set_title('TOFU Forget Quality — forget10 split')
ax.set_ylim(0, 0.75)
ax.legend()
for bar, val in zip(bars, ftr_vals):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.01,
            f'{val:.3f}', ha='center', va='bottom', fontsize=9)
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.savefig(FIGDIR / 'tofu_ftr_bar.png', dpi=150)
plt.show()

## 4. MT-Eval: MTRR Ablation (mt_weight)

In [ ]:
mw_vals   = [0.5, 1.0, 2.0]
mtrr_vals = [0.7075, np.mean([0.69, 0.735, 0.6675]), 0.615]
mtrr_stds = [0.0,    np.std([0.69, 0.735, 0.6675]),  0.0]

fig, ax = plt.subplots(figsize=(6, 4))
ax.errorbar(mw_vals, mtrr_vals, yerr=mtrr_stds,
            marker='o', ms=10, lw=2, color='#1f77b4',
            capsize=6, capthick=2, label='MT-SimNPO (mean ± std, 3 seeds)')

ax.set_xlabel('mt_weight (λ)')
ax.set_ylabel('MTRR (↓ better)')
ax.set_title('MTRR vs mt_weight — MT-Eval val split')
ax.set_xticks(mw_vals)
ax.set_ylim(0.5, 0.85)
ax.legend()

for x, y in zip(mw_vals, mtrr_vals):
    ax.annotate(f'{y:.3f}', (x, y), textcoords='offset points',
                xytext=(8, 4), fontsize=10)
plt.tight_layout()
plt.savefig(FIGDIR / 'mtrr_vs_mw.png', dpi=150)
plt.show()

## 5. MT-Eval: Seed Variance (mw=1.0)

In [ ]:
seeds    = [0, 1, 2]
mtrr_s   = [0.69, 0.735, 0.6675]

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(seeds, mtrr_s, color='#1f77b4', edgecolor='white', width=0.5)
ax.axhline(np.mean(mtrr_s), color='navy', ls='--', lw=1.5,
           label=f'Mean = {np.mean(mtrr_s):.3f}')
ax.set_xlabel('Seed')
ax.set_ylabel('MTRR (↓ better)')
ax.set_title('Seed Variance — MT-SimNPO mw=1.0')
ax.set_xticks(seeds)
ax.set_ylim(0.5, 0.85)
ax.legend()
for s, v in zip(seeds, mtrr_s):
    ax.text(s, v + 0.01, f'{v:.3f}', ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.savefig(FIGDIR / 'mtrr_seed_variance.png', dpi=150)
plt.show()

## 6. Summary Table

In [ ]:
print(f"{'Method':<28} {'FTR':>8} {'MU':>8} {'MTRR':>8}")
print('-' * 56)
rows = [
    ('oracle_retrain',    0.6406, 0.6471, None),
    ('GradDiff',          0.0000, 0.6713, None),
    ('SimNPO',            0.5230, 0.6365, None),
    ('NPO',               None,   None,   None),
    ('MTSimNPO mw=0.5',   0.5242, None,   0.7075),
    ('MTSimNPO mw=1.0',   np.mean([0.5315,0.5266,0.5229]), None, np.mean([0.69,0.735,0.6675])),
    ('MTSimNPO mw=2.0',   0.5241, None,   0.615),
]
for name, ftr, mu, mtrr in rows:
    ftr_s  = f'{ftr:.4f}'  if ftr  is not None else 'TBD'
    mu_s   = f'{mu:.4f}'   if mu   is not None else 'TBD'
    mtrr_s = f'{mtrr:.4f}' if mtrr is not None else 'N/A'
    print(f"{name:<28} {ftr_s:>8} {mu_s:>8} {mtrr_s:>8}")
print('\nNotes:')
print('  FTR  = Forget Truth Ratio (↑ toward oracle 0.64 is better)')
print('  MU   = Model Utility (↑ better)')
print('  MTRR = Multi-Turn Recovery Rate on val split (↓ better)')
print('  NPO training in progress — results TBD')
print('  Vulnerability demo (baselines MTRR) — pending NPO completion')

## 7. Vulnerability Demo Results

*(Run after NPO training completes — see `scripts/run_vulnerability_demo.sh`)*

This section will show MTRR for pre_unlearning, oracle_retrain, GradDiff, NPO, SimNPO on the held-out test split to demonstrate that single-turn unlearning methods remain vulnerable to multi-turn probing.

In [ ]:
# Vulnerability demo results (fill in after running scripts/run_vulnerability_demo.sh)
vuln_results_path = Path('../results/vulnerability')

vuln_methods = ['pre_unlearning', 'oracle_retrain', 'GradDiff', 'NPO', 'SimNPO']
vuln_mtrr = {}
for m in vuln_methods:
    p = vuln_results_path / f'{m}_mt_eval.json'
    if p.exists():
        d = json.loads(p.read_text())
        vuln_mtrr[m] = d.get('overall_mtrr_trained')
    else:
        vuln_mtrr[m] = None

print('Vulnerability Demo — MTRR on test split:')
for m, v in vuln_mtrr.items():
    print(f'  {m:<22} {v:.3f}' if v is not None else f'  {m:<22} (not yet run)')

if any(v is not None for v in vuln_mtrr.values()):
    fig, ax = plt.subplots(figsize=(8, 4))
    names = [m for m, v in vuln_mtrr.items() if v is not None]
    vals  = [vuln_mtrr[m] for m in names]
    vuln_colors = ['#d62728' if m == 'pre_unlearning' else
                   '#2ca02c' if m == 'oracle_retrain' else '#ff7f0e'
                   for m in names]
    ax.bar(names, vals, color=vuln_colors, edgecolor='white', width=0.6)
    ax.set_ylabel('MTRR (↓ better)')
    ax.set_title('Multi-Turn Vulnerability — Baseline Methods (test split)')
    ax.set_ylim(0, 1.05)
    plt.xticks(rotation=15, ha='right')
    plt.tight_layout()
    plt.savefig(FIGDIR / 'vulnerability_demo.png', dpi=150)
    plt.show()
else:
    print('\nRun scripts/run_vulnerability_demo.sh first, then re-run this cell.')